In [1]:
import os
os.getcwd()

'C:\\Users\\justi\\zzzFORTHEFUTURE\\second_project_starwars_etl\\notebook'

In [2]:
import sys
sys.path.append('../')

In [3]:
from pathlib import Path
Path("../logs").mkdir(exist_ok=True)

In [4]:
from src import config_loader
from src import logger_setup

In [5]:
config = config_loader.load_config()
logger_setup.setup_logging(config)

In [6]:
url = config["swapi"]["url"]
endpoints = config["swapi"]["endpoints"]

In [7]:
people_endpoint = url + endpoints[0]
planets_endpoint = url + endpoints[1]
species_endpoint = url + endpoints[2]

In [8]:
people = []
planets = []
species = []

In [9]:
from src import extract

INFO - The path is open : ./data/raw/


In [10]:
extract.extract_data(people_endpoint, people)
extract.extract_data(planets_endpoint, planets)
extract.extract_data(species_endpoint, species)

DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET /api/people HTTP/1.1" 200 None
DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET /api/people/?page=2 HTTP/1.1" 200 None
DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET /api/people/?page=3 HTTP/1.1" 200 None
DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET /api/people/?page=4 HTTP/1.1" 200 None
DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET /api/people/?page=5 HTTP/1.1" 200 None
DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET /api/people/?page=6 HTTP/1.1" 200 None
DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET /api/people/?page=7 HTTP/1.1" 200 None
DEBUG - Starting new HTTPS connection (1): swapi.dev:443
DEBUG - https://swapi.dev:443 "GET

In [11]:
extract.write_raw_data(extract.path, people, 'raw_people')
extract.write_raw_data(extract.path, planets, 'raw_planets')
extract.write_raw_data(extract.path, species, 'raw_species')

INFO - Data loaded
INFO - Data loaded
INFO - Data loaded


In [12]:
from src import transform as t

In [13]:
import pandas as pd

In [14]:
print("For df_people")
print(f"Nb rows: {len(t.people)}")
print(f"Keys: {t.people.keys()}")
print("Shape :", t.people.shape)
print()
for c in t.people.columns :
    if not isinstance(t.people[c], list):
        print(c + ':', pd.api.types.infer_dtype(t.people[c]))
        print(t.people[c].dropna().astype(str).unique()[:5])

For df_people
Nb rows: 82
Keys: Index(['name', 'height', 'mass', 'hair_color', 'skin_color', 'eye_color',
       'birth_year', 'gender', 'homeworld', 'films', 'species', 'vehicles',
       'starships', 'created', 'edited', 'url'],
      dtype='str')
Shape : (82, 16)

name: string
<StringArray>
['Luke Skywalker', 'C-3PO', 'R2-D2', 'Darth Vader', 'Leia Organa']
Length: 5, dtype: str
height: string
<StringArray>
['172', '167', '96', '202', '150']
Length: 5, dtype: str
mass: string
<StringArray>
['77', '75', '32', '136', '49']
Length: 5, dtype: str
hair_color: string
<StringArray>
['blond', 'n/a', 'none', 'brown', 'brown, grey']
Length: 5, dtype: str
skin_color: string
<StringArray>
['fair', 'gold', 'white, blue', 'white', 'light']
Length: 5, dtype: str
eye_color: string
<StringArray>
['blue', 'yellow', 'red', 'brown', 'blue-gray']
Length: 5, dtype: str
birth_year: string
<StringArray>
['19BBY', '112BBY', '33BBY', '41.9BBY', '52BBY']
Length: 5, dtype: str
gender: string
<StringArray>
['mal

In [15]:
t.people.astype(str).nunique()

name          82
height        46
mass          39
hair_color    12
skin_color    30
eye_color     14
birth_year    37
gender         5
homeworld     49
films         19
species       38
vehicles      11
starships     16
created       82
edited        82
url           82
dtype: int64

In [16]:
t.planets['climate'].sort_values().unique()

<StringArray>
[                     'arid',        'arid, rocky, windy',
 'arid, temperate, tropical',     'artificial temperate ',
                    'frigid',                    'frozen',
                       'hot',                'hot, humid',
                     'murky',                  'polluted',
               'superheated',                 'temperate',
           'temperate, arid', 'temperate, arid, subartic',
    'temperate, arid, windy',          'temperate, artic',
          'temperate, moist',       'temperate, tropical',
                  'tropical',       'tropical, temperate',
                   'unknown']
Length: 21, dtype: str

In [17]:
t.species['designation'].unique()

<StringArray>
['sentient', 'reptilian']
Length: 2, dtype: str

In [18]:
t.species.groupby('classification')['classification'].count()

classification
amphibian      6
artificial     1
gastropod      1
insectoid      1
mammal        16
mammals        1
reptile        3
reptilian      1
sentient       1
unknown        6
Name: classification, dtype: int64

In [19]:
t.species['language'].sort_values().unique()

<StringArray>
[        'Aleena',         'Cerean',        'Chagria',       'Clawdite',
           'Dosh',         'Dugese',        'Ewokese', 'Galactic Basic',
 'Galactic basic',  'Galatic Basic',      'Geonosian',   'Gungan basic',
        'Huttese',     'Iktotchese',        'Kaleesh',       'Kaminoan',
        'Kel Dor',       'Mirialan', 'Mon Calamarian',           'Muun',
        'Nautila',      'Neimoidia',       'Quermian',     'Shyriiwook',
        'Skakoan',      'Sullutese',        'Togruti',      'Toydarian',
         'Tundan',       'Twi'leki',        'Utapese',        'Xextese',
        'Zabraki',       'besalisk',            'n/a',        'unknown',
     'vulpterish']
Length: 37, dtype: str

In [20]:
t.people.dtypes

name             str
height           str
mass             str
hair_color       str
skin_color       str
eye_color        str
birth_year       str
gender           str
homeworld        str
films         object
species       object
vehicles      object
starships     object
created          str
edited           str
url              str
dtype: object

In [21]:
t.planets.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   name             60 non-null     str   
 1   rotation_period  60 non-null     str   
 2   orbital_period   60 non-null     str   
 3   diameter         60 non-null     str   
 4   climate          60 non-null     str   
 5   gravity          60 non-null     str   
 6   terrain          60 non-null     str   
 7   surface_water    60 non-null     str   
 8   population       60 non-null     str   
 9   residents        60 non-null     object
 10  films            60 non-null     object
 11  created          60 non-null     str   
 12  edited           60 non-null     str   
 13  url              60 non-null     str   
dtypes: object(2), str(12)
memory usage: 6.7+ KB


In [22]:
t.species.info()

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   name              37 non-null     str   
 1   classification    37 non-null     str   
 2   designation       37 non-null     str   
 3   average_height    37 non-null     str   
 4   skin_colors       37 non-null     str   
 5   hair_colors       37 non-null     str   
 6   eye_colors        37 non-null     str   
 7   average_lifespan  37 non-null     str   
 8   homeworld         36 non-null     str   
 9   language          37 non-null     str   
 10  people            37 non-null     object
 11  films             37 non-null     object
 12  created           37 non-null     str   
 13  edited            37 non-null     str   
 14  url               37 non-null     str   
dtypes: object(2), str(13)
memory usage: 4.5+ KB


In [23]:
t.people.head()

,name,height,mass,hair_color,skin_color,eye_color,birth_year,gender,homeworld,films,species,vehicles,starships,created,edited,url
0,Luke Skywalker,172,77,blond,fair,blue,19BBY,male,https://swapi.dev/api/planets/1/,"[https://swapi.dev/api/films/1/, https://swapi...",[],"[https://swapi.dev/api/vehicles/14/, https://s...","[https://swapi.dev/api/starships/12/, https://...",2014-12-09T13:50:51.644000Z,2014-12-20T21:17:56.891000Z,https://swapi.dev/api/people/1/
1,C-3PO,167,75,n/a,gold,yellow,112BBY,n/a,https://swapi.dev/api/planets/1/,"[https://swapi.dev/api/films/1/, https://swapi...",[https://swapi.dev/api/species/2/],[],[],2014-12-10T15:10:51.357000Z,2014-12-20T21:17:50.309000Z,https://swapi.dev/api/people/2/
2,R2-D2,96,32,n/a,"white, blue",red,33BBY,n/a,https://swapi.dev/api/planets/8/,"[https://swapi.dev/api/films/1/, https://swapi...",[https://swapi.dev/api/species/2/],[],[],2014-12-10T15:11:50.376000Z,2014-12-20T21:17:50.311000Z,https://swapi.dev/api/people/3/
3,Darth Vader,202,136,none,white,yellow,41.9BBY,male,https://swapi.dev/api/planets/1/,"[https://swapi.dev/api/films/1/, https://swapi...",[],[],[https://swapi.dev/api/starships/13/],2014-12-10T15:18:20.704000Z,2014-12-20T21:17:50.313000Z,https://swapi.dev/api/people/4/
4,Leia Organa,150,49,brown,light,brown,19BBY,female,https://swapi.dev/api/planets/2/,"[https://swapi.dev/api/films/1/, https://swapi...",[],[https://swapi.dev/api/vehicles/30/],[],2014-12-10T15:20:09.791000Z,2014-12-20T21:17:50.315000Z,https://swapi.dev/api/people/5/


In [24]:
t.planets.head()

,name,rotation_period,orbital_period,diameter,climate,gravity,terrain,surface_water,population,residents,films,created,edited,url
0,Tatooine,23,304,10465,arid,1 standard,desert,1,200000,"[https://swapi.dev/api/people/1/, https://swap...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-09T13:50:49.641000Z,2014-12-20T20:58:18.411000Z,https://swapi.dev/api/planets/1/
1,Alderaan,24,364,12500,temperate,1 standard,"grasslands, mountains",40,2000000000,"[https://swapi.dev/api/people/5/, https://swap...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T11:35:48.479000Z,2014-12-20T20:58:18.420000Z,https://swapi.dev/api/planets/2/
2,Yavin IV,24,4818,10200,"temperate, tropical",1 standard,"jungle, rainforests",8,1000,[],[https://swapi.dev/api/films/1/],2014-12-10T11:37:19.144000Z,2014-12-20T20:58:18.421000Z,https://swapi.dev/api/planets/3/
3,Hoth,23,549,7200,frozen,1.1 standard,"tundra, ice caves, mountain ranges",100,unknown,[],[https://swapi.dev/api/films/2/],2014-12-10T11:39:13.934000Z,2014-12-20T20:58:18.423000Z,https://swapi.dev/api/planets/4/
4,Dagobah,23,341,8900,murky,N/A,"swamp, jungles",8,unknown,[],"[https://swapi.dev/api/films/2/, https://swapi...",2014-12-10T11:42:22.590000Z,2014-12-20T20:58:18.425000Z,https://swapi.dev/api/planets/5/


In [25]:
t.species.head()

,name,classification,designation,average_height,skin_colors,hair_colors,eye_colors,average_lifespan,homeworld,language,people,films,created,edited,url
0,Human,mammal,sentient,180,"caucasian, black, asian, hispanic","blonde, brown, black, red","brown, blue, green, hazel, grey, amber",120,https://swapi.dev/api/planets/9/,Galactic Basic,"[https://swapi.dev/api/people/66/, https://swa...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T13:52:11.567000Z,2014-12-20T21:36:42.136000Z,https://swapi.dev/api/species/1/
1,Droid,artificial,sentient,n/a,n/a,n/a,n/a,indefinite,NaN,n/a,"[https://swapi.dev/api/people/2/, https://swap...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T15:16:16.259000Z,2014-12-20T21:36:42.139000Z,https://swapi.dev/api/species/2/
2,Wookie,mammal,sentient,210,gray,"black, brown","blue, green, yellow, brown, golden, red",400,https://swapi.dev/api/planets/14/,Shyriiwook,"[https://swapi.dev/api/people/13/, https://swa...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T16:44:31.486000Z,2014-12-20T21:36:42.142000Z,https://swapi.dev/api/species/3/
3,Rodian,sentient,reptilian,170,"green, blue",n/a,black,unknown,https://swapi.dev/api/planets/23/,Galatic Basic,[https://swapi.dev/api/people/15/],[https://swapi.dev/api/films/1/],2014-12-10T17:05:26.471000Z,2014-12-20T21:36:42.144000Z,https://swapi.dev/api/species/4/
4,Hutt,gastropod,sentient,300,"green, brown, tan",n/a,"yellow, red",1000,https://swapi.dev/api/planets/24/,Huttese,[https://swapi.dev/api/people/16/],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T17:12:50.410000Z,2014-12-20T21:36:42.146000Z,https://swapi.dev/api/species/5/


In [26]:
t.df_people.head()

,name,height,mass,hair_color,skin_color,eye_color,birth_year,gender,id_homeworld,id_species
0,Luke Skywalker,172,77,blond,fair,blue,19BBY,male,1,<NA>
1,C-3PO,167,75,NaN,gold,yellow,112BBY,NaN,1,2
2,R2-D2,96,32,NaN,"white, blue",red,33BBY,NaN,8,2
3,Darth Vader,202,136,NaN,white,yellow,41.9BBY,male,1,<NA>
4,Leia Organa,150,49,brown,light,brown,19BBY,female,2,<NA>


In [27]:
t.df_planets.head()

,name,rotation_period,orbital_period,diameter,climate,gravity,terrain,population,id_people
0,Tatooine,23,304,10465,arid,1 standard,desert,200000,1
1,Alderaan,24,364,12500,temperate,1 standard,"grasslands, mountains",2000000000,5
2,Yavin IV,24,4818,10200,"temperate, tropical",1 standard,"jungle, rainforests",1000,<NA>
3,Hoth,23,549,7200,frozen,1.1 standard,"tundra, ice caves, mountain ranges",NaN,<NA>
4,Dagobah,23,341,8900,murky,N/A,"swamp, jungles",NaN,<NA>


In [28]:
t.df_species.head()

,name,classification,designation,average_height,skin_colors,eye_colors,id_homeworld,language,id_people
0,Human,mammal,sentient,180,"caucasian, black, asian, hispanic","brown, blue, green, hazel, grey, amber",9,Galactic Basic,66
1,Droid,artificial,sentient,<NA>,NaN,NaN,<NA>,NaN,2
2,Wookie,mammal,sentient,210,gray,"blue, green, yellow, brown, golden, red",14,Shyriiwook,13
3,Rodian,reptilian,sentient,170,"green, blue",black,23,Galatic Basic,15
4,Hutt,gastropod,sentient,300,"green, brown, tan","yellow, red",24,Huttese,16


In [29]:
t.df_people.info()

<class 'pandas.DataFrame'>
RangeIndex: 82 entries, 0 to 81
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   name          82 non-null     str  
 1   height        81 non-null     Int64
 2   mass          59 non-null     Int64
 3   hair_color    41 non-null     str  
 4   skin_color    81 non-null     str  
 5   eye_color     80 non-null     str  
 6   birth_year    43 non-null     str  
 7   gender        78 non-null     str  
 8   id_homeworld  82 non-null     Int64
 9   id_species    50 non-null     Int64
dtypes: Int64(4), str(6)
memory usage: 6.9 KB


In [30]:
t.df_planets.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   name             59 non-null     str  
 1   rotation_period  48 non-null     Int64
 2   orbital_period   48 non-null     Int64
 3   diameter         44 non-null     Int64
 4   climate          48 non-null     str  
 5   gravity          45 non-null     str  
 6   terrain          53 non-null     str  
 7   population       43 non-null     str  
 8   id_people        49 non-null     Int64
dtypes: Int64(4), str(5)
memory usage: 4.6 KB


In [31]:
t.df_species.info()

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   name            37 non-null     str  
 1   classification  31 non-null     str  
 2   designation     37 non-null     str  
 3   average_height  34 non-null     Int64
 4   skin_colors     36 non-null     str  
 5   eye_colors      34 non-null     str  
 6   id_homeworld    36 non-null     Int64
 7   language        35 non-null     str  
 8   id_people       37 non-null     Int64
dtypes: Int64(3), str(6)
memory usage: 2.8 KB


In [32]:
t.df_people.isnull().sum()

name             0
height           1
mass            23
hair_color      41
skin_color       1
eye_color        2
birth_year      39
gender           4
id_homeworld     0
id_species      32
dtype: int64

In [33]:
t.df_species.isnull().sum()

name              0
classification    6
designation       0
average_height    3
skin_colors       1
eye_colors        3
id_homeworld      1
language          2
id_people         0
dtype: int64

In [34]:
t.df_planets.isnull().sum()

name                1
rotation_period    12
orbital_period     12
diameter           16
climate            12
gravity            15
terrain             7
population         17
id_people          11
dtype: int64

In [35]:
from src import load

In [36]:
load.load(t.df_people, "df_people")
load.load(t.df_planets, "df_planets")
load.load(t.df_species, "df_species")

INFO - df_people.csv saved
INFO - df_planets.csv saved
INFO - df_species.csv saved
